In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from sklearn.model_selection import  train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest,chi2,f_classif
from sklearn.tree import DecisionTreeClassifier

In [ ]:
df = pd.read_csv('/content/sample_data/tested.csv')

In [ ]:
df.shape

(418, 12)

In [ ]:
df.drop(columns=['SibSp','Parch','Ticket','Cabin','Name','PassengerId'],inplace=True)

In [ ]:
df


,Survived,Pclass,Sex,Age,Fare,Embarked
0,0,3,male,34.5,7.8292,Q
1,1,3,female,47.0,7.0000,S
2,0,2,male,62.0,9.6875,Q
3,0,3,male,27.0,8.6625,S
4,1,3,female,22.0,12.2875,S
...,...,...,...,...,...,...
413,0,3,male,NaN,8.0500,S
414,1,1,female,39.0,108.9000,C
415,0,3,male,38.5,7.2500,S
416,0,3,male,NaN,8.0500,S


In [ ]:
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']),
                                                 df['Survived'],test_size=0.2,random_state=42)

In [ ]:
X_train

,Pclass,Sex,Age,Fare,Embarked
336,2,male,32.0,13.0000,S
31,2,male,24.0,31.5000,S
84,2,male,NaN,10.7083,Q
287,1,male,24.0,82.2667,S
317,2,male,19.0,10.5000,S
...,...,...,...,...,...
71,3,male,21.0,7.8958,S
106,3,male,21.0,7.8208,Q
270,1,male,46.0,75.2417,C
348,2,male,24.0,13.5000,S


In [ ]:
X_test

,Pclass,Sex,Age,Fare,Embarked
321,3,male,25.0,7.2292,C
324,1,female,39.0,211.3375,S
388,3,male,21.0,7.7500,Q
56,3,male,35.0,7.8958,S
153,3,female,36.0,12.1833,S
...,...,...,...,...,...
57,3,male,25.0,7.6500,S
126,3,male,22.0,7.7958,S
24,1,female,48.0,262.3750,C
17,3,male,21.0,7.2250,C


In [ ]:
y_train

,Survived
336,0
31,0
84,0
287,0
317,0
...,...
71,0
106,0
270,0
348,0


In [ ]:
y_test

,Survived
321,0
324,1
388,0
56,0
153,1
...,...
57,0
126,0
24,1
17,0


In [ ]:
df.isnull().sum()

,0
Survived,0
Pclass,0
Sex,0
Age,86
Fare,1
Embarked,0


In [ ]:
#imputation transformer
trf1 = ColumnTransformer([
    ('impute_age',SimpleImputer(),[2]),
    ('impute_Fare',SimpleImputer(),[3])],remainder='passthrough')


In [ ]:
#one hot coding
trf2 = ColumnTransformer([
    ('ohe_sex_embarked',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[2,4])

],remainder='passthrough')

In [ ]:
#Scaling
trf3 = ColumnTransformer([
    ('scale',StandardScaler(),slice(0,4))
])

In [ ]:
#Feature Selection
#trf4 = SelectKBest(score_func=chi2,k=4)

In [ ]:
#train the model
trf5 = DecisionTreeClassifier()

In [ ]:
#Create PipeLine
pipe = Pipeline([
    ('trf1',trf1),
    ('trf2',trf2),
    ('trf3',trf3),
    #('trf4',trf4),
    ('trf5',trf5)
])

In [ ]:
#train
pipe.fit(X_train,y_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_Fare',
                                                  SimpleImputer(), [3])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [2, 4])])),
                ('trf3',
                 ColumnTransformer(transformers=[('scale', StandardScaler(),
                                                  slice(0, 4, None))])),
                ('trf5', DecisionTreeClassifier())])

In [ ]:
y_predict = pipe.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_predict)

0.5714285714285714

In [ ]:
#Exporting the Pipeline
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))